# Gold Forecast — Notebook de Ciência de Dados
**Objetivo:** prever o próximo preço observado do ouro com regressão e modelos baseados em árvores.

> Antes de executar: baixe `gold_price.csv` do [Kaggle](https://www.kaggle.com/datasets/lbronchal/gold-and-silver-prices-dataset) e faça upload no Colab. O notebook inspeciona as colunas para que a coluna de preço correta seja escolhida explicitamente.

## 1. Instalação e importação

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from google.colab import files
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path
import os

RANDOM_STATE = 42

## 2. Upload e inspeção dos dados

In [ ]:
uploaded = files.upload()
arquivo = next(iter(uploaded))
df_original = pd.read_csv(arquivo)

print("Dimensões:", df_original.shape)
print("\\nColunas:", df_original.columns.tolist())
display(df_original.head())
df_original.info()

### Escolha das colunas
Veja a saída acima. Ajuste `COLUNA_DATA` e `COLUNA_PRECO` para os nomes exatos do arquivo. Em bases LBMA, pode haver mais de uma cotação ou moeda: escolha uma série única e mantenha a unidade documentada no relatório.

In [ ]:
COLUNA_DATA = "Date"       # ALTERE se o CSV usar outro nome
COLUNA_PRECO = "USD (PM)"  # ALTERE para a coluna de preço correta

assert COLUNA_DATA in df_original.columns, f"Coluna de data não encontrada: {COLUNA_DATA}"
assert COLUNA_PRECO in df_original.columns, f"Coluna de preço não encontrada: {COLUNA_PRECO}"

## 3. Limpeza e análise exploratória

In [ ]:
df = df_original[[COLUNA_DATA, COLUNA_PRECO]].copy()
df.columns = ["Date", "Price"]
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df = df.dropna(subset=["Date", "Price"])
df = df.drop_duplicates(subset=["Date"], keep="last")
df = df.sort_values("Date").reset_index(drop=True)

print("Dimensões após limpeza:", df.shape)
print("Período:", df["Date"].min(), "até", df["Date"].max())
print("Ausentes:\\n", df.isna().sum())
display(df.describe())
display(df.head())

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df["Date"], df["Price"])
plt.title("Histórico do preço do ouro")
plt.xlabel("Data")
plt.ylabel("Preço (unidade conforme dataset)")
plt.grid(alpha=.3)
plt.show()

**Interpretação:** observe tendência, períodos de alta/queda e volatilidade. Não confunda ausência de cotação em fins de semana/feriados com dado faltante: são dias sem observação de mercado. Não vamos inventar preços para esses dias.

## 4. Engenharia de atributos e variável alvo

In [ ]:
FEATURES = [
    "preco_atual", "lag_1", "lag_2", "lag_3", "lag_5", "lag_7",
    "lag_14", "lag_30", "media_7", "media_30",
    "volatilidade_7", "variacao_1"
]

def criar_features(dados):
    d = dados.copy()
    p = d["Price"]
    d["preco_atual"] = p
    for lag in [1, 2, 3, 5, 7, 14, 30]:
        d[f"lag_{lag}"] = p.shift(lag)
    d["media_7"] = p.rolling(7).mean()
    d["media_30"] = p.rolling(30).mean()
    d["volatilidade_7"] = p.rolling(7).std()
    d["variacao_1"] = p.pct_change()
    d["target"] = p.shift(-1)
    return d.dropna().reset_index(drop=True)

dados = criar_features(df)
display(dados[["Date"] + FEATURES + ["target"]].head())
print("Linhas após criação das features:", len(dados))

**Por que `shift(-1)` no target?** A linha da observação atual recebe como alvo o preço da próxima observação. As features são construídas com dados disponíveis até a data da linha, sem olhar o futuro.

## 5. Separação temporal: treino e teste

In [ ]:
corte = int(len(dados) * 0.80)
treino = dados.iloc[:corte].copy()
teste = dados.iloc[corte:].copy()

X_treino, y_treino = treino[FEATURES], treino["target"]
X_teste, y_teste = teste[FEATURES], teste["target"]

print("Treino:", treino["Date"].min(), "até", treino["Date"].max(), X_treino.shape)
print("Teste:", teste["Date"].min(), "até", teste["Date"].max(), X_teste.shape)

Não embaralhamos a série. O teste contém as observações mais recentes, simulando o uso do passado para prever o futuro. A avaliação é de um passo à frente: em cada data do teste, os preços anteriores já observados podem ser usados para calcular os lags.

## 6. Treinamento e validação temporal dos modelos

In [ ]:
modelos = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=8, min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=250, max_depth=14, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    )
}

tscv = TimeSeriesSplit(n_splits=5)
resultados_cv = []

for nome, modelo in modelos.items():
    scores = cross_val_score(
        modelo, X_treino, y_treino,
        cv=tscv, scoring="neg_mean_absolute_error"
    )
    resultados_cv.append({
        "Modelo": nome,
        "MAE_CV": -scores.mean(),
        "Desvio_CV": scores.std()
    })

cv_df = pd.DataFrame(resultados_cv).sort_values("MAE_CV")
display(cv_df)

A validação cruzada temporal é feita apenas dentro do conjunto de treino. Selecionamos o algoritmo com menor MAE médio de validação, sem usar o conjunto de teste para escolher o vencedor.

## 7. Avaliação final no teste e baseline

In [ ]:
melhor_nome = cv_df.iloc[0]["Modelo"]
melhor_modelo = modelos[melhor_nome]
melhor_modelo.fit(X_treino, y_treino)

y_pred = melhor_modelo.predict(X_teste)

def metricas_regressao(y_real, y_previsto):
    return {
        "MAE": mean_absolute_error(y_real, y_previsto),
        "MSE": mean_squared_error(y_real, y_previsto),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_previsto)),
        "R2": r2_score(y_real, y_previsto)
    }

metricas_modelo = metricas_regressao(y_teste, y_pred)
# Baseline: repetir o último preço conhecido.
y_baseline = teste["preco_atual"].to_numpy()
metricas_baseline = metricas_regressao(y_teste, y_baseline)

comparacao = pd.DataFrame([
    {"Modelo": melhor_nome, **metricas_modelo},
    {"Modelo": "Baseline (último preço)", **metricas_baseline}
])
display(comparacao)

## 8. Gráfico: real x previsto

In [ ]:
resultado_teste = pd.DataFrame({
    "Date": teste["Date"].to_numpy(),
    "Real": y_teste.to_numpy(),
    "Previsto": y_pred
})

plt.figure(figsize=(15,5))
plt.plot(resultado_teste["Date"], resultado_teste["Real"], label="Real")
plt.plot(resultado_teste["Date"], resultado_teste["Previsto"], label="Previsto")
plt.title("Preço real x previsto — conjunto de teste")
plt.xlabel("Data")
plt.ylabel("Preço")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**Como interpretar:** MAE e RMSE estão na mesma unidade do preço; quanto menores, melhor. MSE penaliza mais os erros grandes. R² próximo de 1 indica que o modelo explica boa parte da variação; pode ser negativo. Compare o modelo com o baseline: se não superar a referência simples, não devemos afirmar que ele agrega poder preditivo.

## 9. Persistência do modelo e artefatos

In [ ]:
from pathlib import Path
Path("models").mkdir(exist_ok=True)

bundle = {
    "model": melhor_modelo,
    "model_name": melhor_nome,
    "features": FEATURES,
    "history": df,
    "price_column_original": COLUNA_PRECO,
    "date_column_original": COLUNA_DATA,
    "metrics": {k: float(v) for k, v in metricas_modelo.items()},
    "baseline_metrics": {k: float(v) for k, v in metricas_baseline.items()},
    "cv_results": cv_df,
    "test_predictions": resultado_teste,
    "training_cutoff": str(treino["Date"].max().date()),
    "test_start": str(teste["Date"].min().date()),
    "test_end": str(teste["Date"].max().date())
}

joblib.dump(bundle, "models/gold_forecast_bundle.joblib")
cv_df.to_csv("models/comparacao_validacao_temporal.csv", index=False)
comparacao.to_csv("models/metricas_teste.csv", index=False)
resultado_teste.to_csv("models/previsoes_teste.csv", index=False)
df.to_csv("models/historico_limpo.csv", index=False)

print("Modelo selecionado:", melhor_nome)
print("Artefatos salvos.")

## 10. Download do modelo para usar no Streamlit

In [ ]:
!zip -r gold_forecast_modelos.zip models
files.download("gold_forecast_modelos.zip")

## Conclusão do notebook

Registre no relatório os valores efetivamente calculados de MAE, MSE, RMSE e R², o modelo escolhido, as datas do treino/teste e se o modelo superou o baseline. Não invente resultados: eles dependem do CSV e da coluna de preço selecionada.